In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Path to your zip file in Drive
zip_path = "/content/drive/My Drive/data.zip"   # adjust if it's inside a folder

# 3. Extract into /content (root of Colab)
import zipfile, os

extract_path = "/content"   # ensures "data/" is created at /content/data

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 4. Verify that "data" folder exists at /content/data
print(os.listdir("/content"))
print(os.listdir("/content/data"))


In [1]:
import tensorflow as tf
import numpy as np
from config import (
    CLASS_IDS,
    TRAIN_IMG_DIR,
    TRAIN_MASK_DIR,
    CLASS_LABELS,
    VAL_IMG_DIR,
    VAL_MASK_DIR,
    TEST_IMG_DIR,
    TEST_MASK_DIR,
    BATCH_SIZE,
    WEIGHT_MAX_CAP_LOG_INVERSE_FREQUENCY,
    LEARNING_RATE,
    EPOCHS,
 )
from data_preprocess_pipeline import build_dataset,remap_mask,get_sorted_file_paths
from model import make_Unet_with_resnet50
from collections import defaultdict

In [ ]:
import os

SAVE_DIR = "/content/drive/My Drive/unet_v4"
os.makedirs(SAVE_DIR, exist_ok=True)

LAST_MODEL_PATH = os.path.join(SAVE_DIR, "last_model.keras")
BEST_MODEL_PATH = os.path.join(SAVE_DIR, "best_model.keras")
LOG_FILE_PATH   = os.path.join(SAVE_DIR, "log.txt")


In [2]:
TRAIN_CLASS_FREQ = [
    0.03529683,  # Trees
    0.05933981,  # Lush Bushes
    0.18872895,  # Dry Grass
    0.01099030,  # Dry Bushes
    0.04394827,  # Ground Clutter
    0.02807256,  # Flower
    0.00078129,  # Logs
    0.01198249,  # Rocks
    0.24447772,  # Landscape
    0.37638177,  # Sky
]
print(CLASS_LABELS)
print(TRAIN_CLASS_FREQ)

['Trees', 'Lush Bushes', 'Dry Grass', 'Dry Bushes', 'Ground Clutter', 'Flower', 'Logs', 'Rocks', 'Landscape', 'Sky']
[0.03529683, 0.05933981, 0.18872895, 0.0109903, 0.04394827, 0.02807256, 0.00078129, 0.01198249, 0.24447772, 0.37638177]


In [3]:
freq=np.array(TRAIN_CLASS_FREQ,dtype=np.float32)
weights=1.0/np.log(1.02+freq)
print("unclipped weights:",weights)
weights=np.clip(weights,1.0,WEIGHT_MAX_CAP_LOG_INVERSE_FREQUENCY)
print("clipped weights",weights)
CLASS_WEIGHTS=tf.constant(weights,tf.float32)
print(CLASS_WEIGHTS.numpy())

unclipped weights: [18.579752  13.097661   5.275115  32.76568   16.132473  21.297968
 48.618523  31.764551   4.2615     2.9950485]
clipped weights [5.        5.        5.        5.        5.        5.        5.
 5.        4.2615    2.9950485]
[5.        5.        5.        5.        5.        5.        5.
 5.        4.2615    2.9950485]


In [4]:
def _safe_register_keras_serializable(package, name=None):
    def decorator(obj):
        reg_name = f"{package}>{name or obj.__name__}"
        if reg_name in tf.keras.utils.get_custom_objects():
            return obj
        return tf.keras.utils.register_keras_serializable(
            package=package, name=name
        )(obj)
    return decorator


@_safe_register_keras_serializable(package="Custom")
def weighted_sparse_ce(y_true, y_pred):
    """
    y_true: (B, H, W) integer class IDs
    y_pred: (B, H, W, C) softmax probabilities
    """

    # Ensure y_true is int32 for indexing
    y_true = tf.cast(y_true, tf.int32)

    # Compute standard sparse categorical crossentropy
    ce = tf.keras.losses.sparse_categorical_crossentropy(
        y_true=y_true, y_pred=y_pred
    )

    # Gather weights per pixel based on class id
    weights = tf.gather(CLASS_WEIGHTS, y_true)

    # Apply weights
    weighted_ce = ce * weights
    return tf.reduce_mean(weighted_ce)


@_safe_register_keras_serializable(package="Custom")
def dice_loss(y_true, y_pred, smooth=1e-6):
    """
    Multi-class Dice loss computed over all pixels.
    """
    y_true = tf.cast(y_true, tf.int32)
    y_pred = tf.cast(y_pred, tf.float32)

    num_classes = tf.shape(y_pred)[-1]
    y_true_oh = tf.one_hot(y_true, depth=num_classes, dtype=tf.float32)

    axes = [0, 1, 2]
    intersection = tf.reduce_sum(y_true_oh * y_pred, axis=axes)
    denominator = tf.reduce_sum(y_true_oh + y_pred, axis=axes)
    dice = (2.0 * intersection + smooth) / (denominator + smooth)

    return 1.0 - tf.reduce_mean(dice)


@_safe_register_keras_serializable(package="Custom")
def total_loss(y_true, y_pred):
    return weighted_sparse_ce(y_true, y_pred) + 0.7 * dice_loss(y_true, y_pred)

In [5]:
def pred_labels(y_pred):
    # y_pred: (B,H,W,C)
    return tf.argmax(y_pred,axis=-1,output_type=tf.int32)

def per_class_iou(y_true,y_pred,num_classes=len(CLASS_IDS)):
    """
    Returns a tensor of shape (num_classes,)
    """
    y_pred=pred_labels(y_pred=y_pred)
    y_true=tf.cast(y_true,tf.int32)

    ious=[]
    for c in range(num_classes):
        true_c=tf.equal(y_true,c)
        pred_c=tf.equal(y_pred,c)
        intersection=tf.reduce_sum(
            tf.cast(true_c&pred_c,tf.float32)
        )
        union=tf.reduce_sum(
            tf.cast(true_c|pred_c,tf.float32)
        )
        iou=tf.math.divide_no_nan(intersection,union)
        ious.append(iou)

    return tf.stack(ious)

def mean_iou_metric(y_true,y_pred,num_classes=len(CLASS_IDS)):
    ious=per_class_iou(y_true=y_true,y_pred=y_pred,num_classes=num_classes)
    return tf.reduce_mean(ious)

def worst_iou_metric(y_true, y_pred,num_classes=len(CLASS_IDS)):
    ious = per_class_iou(y_true, y_pred, num_classes=num_classes)
    return tf.reduce_min(ious)

def make_class_iou_metric(class_id):
    label = CLASS_LABELS[class_id]
    def class_iou(y_true, y_pred):
        ious = per_class_iou(y_true, y_pred, num_classes=len(CLASS_IDS))
        return ious[class_id]
    class_iou.__name__ = f"iou_{label}"
    return class_iou

metrics = [
    mean_iou_metric,
    worst_iou_metric,
]

for i in range(len(CLASS_IDS)):
    metrics.append(make_class_iou_metric(i))

# DATASET LEVEL IOU CALCULATION (ACCUMULATED OVER EVERY BATCH)

In [6]:
def _safe_register_keras_serializable(package, name=None):
    def decorator(obj):
        reg_name = f"{package}>{name or obj.__name__}"
        if reg_name in tf.keras.utils.get_custom_objects():
            return obj
        return tf.keras.utils.register_keras_serializable(
            package=package, name=name
        )(obj)
    return decorator


@_safe_register_keras_serializable(package="Custom")
class DatasetIoU(tf.keras.metrics.Metric):
    def __init__(self, num_classes, name="dataset_iou", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes

        self.tp = self.add_weight(
            shape=(num_classes,), initializer="zeros", name="tp"
        )
        self.fp = self.add_weight(
            shape=(num_classes,), initializer="zeros", name="fp"
        )
        self.fn = self.add_weight(
            shape=(num_classes,), initializer="zeros", name="fn"
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        y_true = tf.cast(y_true, tf.int32)

        # Compute per-class TP / FP / FN in vectorized form
        tp_list = []
        fp_list = []
        fn_list = []

        for c in range(self.num_classes):
            true_c = tf.equal(y_true, c)
            pred_c = tf.equal(y_pred, c)

            tp = tf.reduce_sum(tf.cast(true_c & pred_c, tf.float32))
            fp = tf.reduce_sum(tf.cast(~true_c & pred_c, tf.float32))
            fn = tf.reduce_sum(tf.cast(true_c & ~pred_c, tf.float32))

            tp_list.append(tp)
            fp_list.append(fp)
            fn_list.append(fn)

        # Stack into tensors of shape (num_classes,)
        tp_vec = tf.stack(tp_list)
        fp_vec = tf.stack(fp_list)
        fn_vec = tf.stack(fn_list)

        # SAFE variable updates
        self.tp.assign_add(tp_vec)
        self.fp.assign_add(fp_vec)
        self.fn.assign_add(fn_vec)

    def reset_state(self):
        self.tp.assign(tf.zeros_like(self.tp))
        self.fp.assign(tf.zeros_like(self.fp))
        self.fn.assign(tf.zeros_like(self.fn))

    def get_config(self):
        config = super().get_config()
        config.update({"num_classes": self.num_classes})
        return config


@_safe_register_keras_serializable(package="Custom")
class DatasetMeanIoU(DatasetIoU):
    def __init__(self, num_classes, name="dataset_mean_iou", **kwargs):
        super().__init__(num_classes=num_classes, name=name, **kwargs)

    def result(self):
        iou = tf.math.divide_no_nan(
            self.tp, self.tp + self.fp + self.fn
        )
        return tf.reduce_mean(iou)

    def get_config(self):
        config = super().get_config()
        return config


@_safe_register_keras_serializable(package="Custom")
class DatasetMeanIoUPresentOnly(DatasetIoU):
    def __init__(self, num_classes, name="dataset_mean_iou_present_only", **kwargs):
        super().__init__(num_classes=num_classes, name=name, **kwargs)
    def result(self):
        # Count a class if it exists in GT or the model predicts it (penalize false positives).
        gt_pixels = self.tp + self.fn
        pred_pixels = self.tp + self.fp
        present = tf.logical_or(tf.greater(gt_pixels, 0), tf.greater(pred_pixels, 0))
        iou = tf.math.divide_no_nan(self.tp, self.tp + self.fp + self.fn)
        present_iou = tf.boolean_mask(iou, present)
        return tf.cond(
            tf.size(present_iou) > 0,
            lambda: tf.reduce_mean(present_iou),
            lambda: tf.constant(0.0, dtype=iou.dtype),
        )

    def get_config(self):
        config = super().get_config()
        return config


@_safe_register_keras_serializable(package="Custom")
class DatasetWorstIoU(DatasetIoU):
    def __init__(self, num_classes, name="dataset_worst_iou", **kwargs):
        super().__init__(num_classes=num_classes, name=name, **kwargs)

    def result(self):
        iou = tf.math.divide_no_nan(
            self.tp, self.tp + self.fp + self.fn
        )
        return tf.reduce_min(iou)

    def get_config(self):
        config = super().get_config()
        return config


@_safe_register_keras_serializable(package="Custom")
class DatasetClassIoU(DatasetIoU):
    def __init__(self, class_index, class_name, num_classes=10, **kwargs):
        super().__init__(num_classes=num_classes, name=f"dataset_iou_{class_name}", **kwargs)
        self.class_index = class_index
        self.class_name = class_name

    def result(self):
        tp = self.tp[self.class_index]
        fp = self.fp[self.class_index]
        fn = self.fn[self.class_index]
        return tf.math.divide_no_nan(tp, tp + fp + fn)

    def get_config(self):
        config = super().get_config()
        config.update({
            "class_index": self.class_index,
            "class_name": self.class_name,
        })
        return config

# BATCH LEVEL IOU CALCULATION FOR DEBUGGING

In [7]:
def batch_mean_iou_skip_absent(y_true, y_pred, num_classes):
    y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    y_true = tf.cast(y_true, tf.int32)

    ious = []

    for c in range(num_classes):
        true_c = tf.equal(y_true, c)
        pred_c = tf.equal(y_pred, c)

        union = tf.reduce_sum(tf.cast(true_c | pred_c, tf.float32))
        intersection = tf.reduce_sum(tf.cast(true_c & pred_c, tf.float32))

        # skip classes absent in GT AND prediction
        if tf.equal(union, 0):
            continue

        iou = tf.math.divide_no_nan(intersection, union)
        ious.append(iou)

    return tf.reduce_mean(ious) if ious else tf.constant(0.0)

def batch_class_iou(y_true, y_pred, class_index):
    y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    y_true = tf.cast(y_true, tf.int32)

    true_c = tf.equal(y_true, class_index)
    pred_c = tf.equal(y_pred, class_index)

    union = tf.reduce_sum(tf.cast(true_c | pred_c, tf.float32))
    intersection = tf.reduce_sum(tf.cast(true_c & pred_c, tf.float32))

    iou = tf.math.divide_no_nan(intersection, union)
    class_present = tf.greater(union, 0)

    return iou, class_present


metrics = [
    DatasetMeanIoUPresentOnly(len(CLASS_IDS)),
    DatasetMeanIoU(len(CLASS_IDS)),
    DatasetWorstIoU(len(CLASS_IDS)),
]

for i, label in enumerate(CLASS_LABELS):
    metrics.append(
        DatasetClassIoU(i, label, len(CLASS_IDS))
    )

In [8]:
class DebugDatasetWrapper:
    def __init__(self, dataset):
        self.dataset = dataset
        self.last_batch = None

    def generator(self):
        for batch in self.dataset:
            self.last_batch = batch
            yield batch

    def get_dataset(self):
        return tf.data.Dataset.from_generator(
            self.generator,
            output_signature=self.dataset.element_spec
        )

class BatchDebugIoUCallback(tf.keras.callbacks.Callback):
    def __init__(self, debug_dataset, num_classes, class_labels):
        super().__init__()
        self.debug_dataset = debug_dataset
        self.num_classes = num_classes
        self.class_labels = class_labels

    def on_train_batch_end(self, batch, logs=None):
        x, y_true = self.debug_dataset.last_batch
        y_pred = self.model(x, training=False)

        mean_iou = batch_mean_iou_skip_absent(
            y_true, y_pred, self.num_classes
        )

        print(f"\n[Batch Debug] mean IoU (skip absent): {mean_iou.numpy():.4f}")

        for i, label in enumerate(self.class_labels):
            iou,class_present = batch_class_iou(y_true, y_pred, i)
            print(f"  {label:15s}: {iou.numpy(): 4f} | present:{class_present}")


In [9]:
train_dataset=build_dataset(TRAIN_IMG_DIR,TRAIN_MASK_DIR,isTrainingDataset=True,performAugment=True)
val_dataset=build_dataset(VAL_IMG_DIR,VAL_MASK_DIR,isTrainingDataset=False,performAugment=False)

In [10]:
debug_train_wrapper = DebugDatasetWrapper(
    train_dataset.take(10)
)
debug_train_dataset = debug_train_wrapper.get_dataset()


In [ ]:
model=make_Unet_with_resnet50()

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=total_loss,
    metrics=metrics
)



In [11]:
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger

# Save last model after each epoch
checkpoint_last = ModelCheckpoint(
    filepath=LAST_MODEL_PATH,
    save_weights_only=False,
    save_best_only=False,
    verbose=1
)

# Save best model (based on validation mean IoU)
checkpoint_best = ModelCheckpoint(
    filepath=BEST_MODEL_PATH,
    save_weights_only=False,
    save_best_only=True,
    monitor="val_dataset_mean_iou_present_only",  # metric name from DatasetMeanIoUPresentOnly
    mode="max",
    verbose=1
)

# Log training progress to log.txt in Drive
csv_logger = CSVLogger(LOG_FILE_PATH, append=True)
class PeriodicLogCallback(tf.keras.callbacks.Callback):
    def __init__(self, log_file, every_n_epochs=5):
        super().__init__()
        self.log_file = log_file
        self.every_n_epochs = every_n_epochs

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n_epochs == 0:
            with open(self.log_file, "a") as f:
                f.write(f"Epoch {epoch+1}: {logs}\n")
            print(f"[PeriodicLog] Epoch {epoch+1}: logs written to {self.log_file}")


NameError: name 'LAST_MODEL_PATH' is not defined

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[
        checkpoint_last,
        checkpoint_best,
        csv_logger,
        PeriodicLogCallback(LOG_FILE_PATH, every_n_epochs=5)
    ],
)


In [ ]:
model.save("abcd.h5",include_optimizer=True)

In [ ]:
model.save("abcd_wo_opt.h5")
model.save("abcd.keras")

In [12]:
test_dataset=build_dataset(TEST_IMG_DIR,TEST_MASK_DIR,isTrainingDataset=False,performAugment=False)

In [ ]:
import os
import zipfile

MODEL_PATH = "to_test.keras"

custom_objects = {
    "weighted_sparse_ce": weighted_sparse_ce,
    "dice_loss": dice_loss,
    "total_loss": total_loss,
    "DatasetIoU": DatasetIoU,
    "DatasetMeanIoU": DatasetMeanIoU,
    "DatasetMeanIoUPresentOnly": DatasetMeanIoUPresentOnly,
    "DatasetWorstIoU": DatasetWorstIoU,
    "DatasetClassIoU": DatasetClassIoU,
}

def load_any_model(path, custom_objects):
    if not tf.io.gfile.exists(path):
        raise FileNotFoundError(f"Model file not found: {path}")
    is_zip = zipfile.is_zipfile(path)
    ext = os.path.splitext(path)[1].lower()
    if ext == ".keras" and is_zip:
        try:
            import keras
            if hasattr(keras, "saving") and hasattr(keras.saving, "load_model"):
                return keras.saving.load_model(
                    path, custom_objects=custom_objects, compile=False
                )
            if hasattr(keras, "models") and hasattr(keras.models, "load_model"):
                return keras.models.load_model(
                    path, custom_objects=custom_objects, compile=False
                )
        except Exception:
            pass
        h5_path = os.path.splitext(path)[0] + ".h5"
        if tf.io.gfile.exists(h5_path):
            return tf.keras.models.load_model(
                h5_path, custom_objects=custom_objects, compile=False
            )
        raise RuntimeError(
            "Unable to load .keras zip model with this TensorFlow/Keras. "
            "Re-save as .h5 (e.g., in Colab) and set MODEL_PATH to that file."
        )
    return tf.keras.models.load_model(
        path, custom_objects=custom_objects, compile=False
    )

model = load_any_model(MODEL_PATH, custom_objects)
metric = DatasetMeanIoUPresentOnly(len(CLASS_IDS))

for images, masks in test_dataset:
    preds = model(images, training=False)
    metric.update_state(masks, preds)

print("Test Dataset Mean IoU (present only):", metric.result().numpy())


RuntimeError: Unable to load .keras zip model. Install keras>=3 or re-save as .h5.

In [ ]:
# Evaluate the loaded model on the test set
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=total_loss,
    metrics=metrics
)

eval_results = model.evaluate(test_dataset, return_dict=True)
print("Eval results:", eval_results)

In [ ]:
import matplotlib.pyplot as plt

def compute_per_class_iou_single(y_true, y_pred, num_classes):
    ious = []
    for c in range(num_classes):
        true_c = (y_true == c)
        pred_c = (y_pred == c)

        intersection = np.logical_and(true_c, pred_c).sum()
        union = np.logical_or(true_c, pred_c).sum()

        if union == 0:
            ious.append(np.nan)  # skip absent classes
        else:
            ious.append(intersection / union)

    return np.array(ious)


def visualize_prediction(model, dataset, class_labels):
    # Take one batch
    for images, masks in dataset.take(1):
        break

    image = images[0:1]  # keep batch dim
    true_mask = masks[0].numpy()

    # Predict
    pred = model.predict(image)
    pred_mask = np.argmax(pred[0], axis=-1)

    # Compute error %
    pixel_error = np.mean(pred_mask != true_mask) * 100

    # Compute per-class IoU
    ious = compute_per_class_iou_single(
        true_mask,
        pred_mask,
        num_classes=len(class_labels)
    )

    # Lowest IoU class
    valid_ious = np.where(np.isnan(ious), 1.0, ious)
    worst_class_idx = np.argmin(valid_ious)
    worst_class_name = class_labels[worst_class_idx]

    # ===============================
    # Plot
    # ===============================
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow((image[0].numpy() + [103.939,116.779,123.68])[:,:,::-1] / 255.0)
    axes[0].set_title("RGB Image")

    axes[1].imshow(true_mask, cmap="tab20")
    axes[1].set_title("Ground Truth Mask")

    axes[2].imshow(pred_mask, cmap="tab20")
    axes[2].set_title("Predicted Mask")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(
        f"Pixel Error: {pixel_error:.2f}% | "
        f"Worst Class: {worst_class_name} (IoU={ious[worst_class_idx]:.4f})"
    )

    plt.show()

    # Print full class IoUs
    print("\nPer-class IoU:")
    for label, iou in zip(class_labels, ious):
        print(f"{label:15s}: {iou:.4f}")

    print(f"\nPixel Error %: {pixel_error:.2f}%")
    print(f"Worst Class: {worst_class_name}")


In [ ]:
visualize_prediction(model, test_dataset, CLASS_LABELS)
